**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Prepare

**Check data**

In [2]:
ls ${FD_RES}/analysis_variant_motif_richard/motif*lods.pkl

/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl


**Check module**

In [3]:
${FP_APP} python - <<'PY'
import motifdelta, pkgutil
print("motifdelta path:", list(motifdelta.__path__))
print("submodules:", [m.name for m in pkgutil.iter_modules(motifdelta.__path__)])
PY

motifdelta path: ['/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/motifdelta/src/motifdelta']
submodules: ['score', 'seq', 'stats']


In [4]:
${FP_APP} python -c "from motifdelta.stats import precompute_pmaps; print('OK', precompute_pmaps)"

OK <function precompute_pmaps at 0x7f29f9acdd00>


**Set compute resource**

In [5]:
### choose between biostat and igvf
CHOOSE_PARTITION="biostat"
#CHOOSE_PARTITION="igvf"

if [[ "$CHOOSE_PARTITION" == "igvf" ]]; then
    SLURM_ACCOUNT="majoroslab"
    SLURM_PARTITION="igvf,common"
elif [[ "$CHOOSE_PARTITION" == "biostat" ]]; then
    SLURM_ACCOUNT="biostat"
    SLURM_PARTITION="biostat"
else
    echo "Unknown CHOOSE_PARTITION: $CHOOSE_PARTITION"
    exit 1
fi

echo ${SLURM_ACCOUNT}
echo ${SLURM_PARTITION}

biostat
biostat


## Jaspar 2024

### Execute

In [6]:
TXT_FDIRY_INP="${FD_RES}/analysis_variant_motif_richard"
TXT_FNAME_INP="motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl"
TXT_FPATH_INP=${TXT_FDIRY_INP}/${TXT_FNAME_INP}

TXT_FDIRY_OUT=${TXT_FDIRY_INP}
TXT_FNAME_OUT_PMAP="motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl"
TXT_FPATH_OUT_PMAP=${TXT_FDIRY_OUT}/${TXT_FNAME_OUT_PMAP}
TXT_FNAME_OUT_BIND="motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl"
TXT_FPATH_OUT_BIND=${TXT_FDIRY_OUT}/${TXT_FNAME_OUT_BIND}

ls -lh ${TXT_FPATH_INP}

-rw-r--r--. 1 kk319 majoroslab 627K Feb 23 18:12 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl


In [7]:
### set script
FP_EXE=${FD_EXE}/run_motif_lods2pmap.py

### set log file
FP_LOG=${FD_LOG}/run_motif_lods2pmap_jaspar2024.txt

### set resource
NUM_CPU=51
NUM_MEM=10G

SLURM_OPTS=(
  -A "${SLURM_ACCOUNT}"
  -p "${SLURM_PARTITION}"
  --job-name=lods2pmap_jaspar2024
  --cpus-per-task="${NUM_CPU}"
  --mem="${NUM_MEM}"
  --output="${FP_LOG}"
  --chdir="${FD_EXE}"
  --export=ALL,FP_CNF="${FP_CNF}"
  --parsable
)

### execute
SLURM_JOBID=$(sbatch "${SLURM_OPTS[@]}" <<EOF
#!/bin/bash
set -euo pipefail

### init
timer_start=\$(date +%s)
source "${FP_CNF}"

### print start message
echo "Hostname:   \$(hostname)"
echo "Time Stamp: \$(date +"%m-%d-%y+%T")" 
echo "PWD:    \$(pwd)"
echo "FP_APP: ${FP_APP}"
echo "FD_EXE: ${FD_EXE}"
echo "PYTHONPATH (host): \${PYTHONPATH:-<empty>}"
echo

### execute
echo "=== Run main script ==="
${FP_APP} python ${FP_EXE} \
    --txt_fpath_inp       "${TXT_FPATH_INP}" \
    --txt_fpath_out_pmap  "${TXT_FPATH_OUT_PMAP}" \
    --txt_fpath_out_tbind "${TXT_FPATH_OUT_BIND}" \
    --num_workers 50

### print end message
timer=\$(date +%s)
runtime=\$(( timer - timer_start ))
echo
echo 'Done!'
echo "Run Time: \$(displaytime \${runtime})"
EOF
)

echo "Submitted job: ${SLURM_JOBID}"

Submitted job: 43673855


### Review

In [8]:
sacct_summary.sh ${SLURM_JOBID}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43673855.ba+                          batch  COMPLETED   00:00:13  00:10.357    621752K                 dcc-biostat-18 

===== ElapsedRaw =====
ElapsedRaw = 13 sec (0.22 min)

===== MaxRSS =====
MaxRSS = 0.59 GiB


In [9]:
cat ${FD_LOG}/run_motif_lods2pmap_jaspar2024.txt

Hostname:   dcc-biostat-18
Time Stamp: 02-23-26+18:17:09
PWD:    /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
FP_APP: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_script.sh
FD_EXE: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PYTHONPATH (host): <empty>

=== Run main script ===
Loaded /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Loaded 879 motifs
Loaded background: [0.31068275 0.18781467 0.18880767 0.31269491]
[100/879] MA0140.3 GATA1::TAL1
[200/879] MA0602.2 Arid5a
[300/879] MA0655.1 JDP2
[400/879] MA0804.2 TBX19
[500/879] MA0903.2 HOXB3
[600/879] MA1498.3 HOXA7
[700/879] MA1557.1 SMAD5
[800/879] MA1961.2 PATZ1
[879/879] MA2341.1 FEZF2
Finished precomputing p-maps for 879/879 motifs.
Sanity motif: MA0002.3 Runx1 grid_len= 12001 Tbind= 6.619999999986746
Saved pmap motifs: 879 -> /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_ri

In [10]:
${FP_APP} python - <<'PY'
import pickle, numpy as np

fp_pmap = "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl"
fp_tbind = "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl"

with open(fp_pmap,"rb") as f: pmap = pickle.load(f)
with open(fp_tbind,"rb") as f: tb = pickle.load(f)

print("pmap keys:", pmap.keys())
print("meta:", pmap["meta"])
print("num motifs:", len(pmap["motifs"]))
k = next(iter(pmap["motifs"]))
m = pmap["motifs"][k]
print("example:", k, "grid", len(m["arr_num_score_grid"]), "pmf sum", np.sum(m["arr_num_score_pmf"]), "ccdf[0]", m["arr_num_score_ccdf"][0], "tbind", m["num_Tbind"])

print("tbind keys:", tb.keys())
print("num tbind:", len(tb["tbind"]))
PY

pmap keys: dict_keys(['meta', 'motifs'])
meta: {'alphabet': 'ACGT', 'bg': array([0.31068275, 0.18781467, 0.18880767, 0.31269491]), 'alpha': 0.001, 'precision': 0.01, 'score_range': (-60, 60)}
num motifs: 879
example: MA0002.3 Runx1 grid 12001 pmf sum 1.0 ccdf[0] 1.0 tbind 6.619999999986746
tbind keys: dict_keys(['meta', 'tbind'])
num tbind: 879


## Non-redundant motifs

### Execute

In [11]:
TXT_FDIRY_INP="${FD_RES}/analysis_variant_motif_richard"
TXT_FNAME_INP="motif_nonredundant_jvierstra_v2.1beta.lods.pkl"
TXT_FPATH_INP=${TXT_FDIRY_INP}/${TXT_FNAME_INP}

TXT_FDIRY_OUT=${TXT_FDIRY_INP}
TXT_FNAME_OUT_PMAP="motif_nonredundant_jvierstra_v2.1beta.pmap.pkl"
TXT_FPATH_OUT_PMAP=${TXT_FDIRY_OUT}/${TXT_FNAME_OUT_PMAP}
TXT_FNAME_OUT_BIND="motif_nonredundant_jvierstra_v2.1beta.tbind.pkl"
TXT_FPATH_OUT_BIND=${TXT_FDIRY_OUT}/${TXT_FNAME_OUT_BIND}

ls -lh ${TXT_FPATH_INP}

-rw-r--r--. 1 kk319 majoroslab 562K Feb 23 18:13 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl


In [12]:
### set script
FP_EXE=${FD_EXE}/run_motif_lods2pmap.py

### set log file
FP_LOG=${FD_LOG}/run_motif_lods2pmap_jvierstra.txt

### set resource
NUM_CPU=51
NUM_MEM=10G

SLURM_OPTS=(
  -A "${SLURM_ACCOUNT}"
  -p "${SLURM_PARTITION}"
  --job-name=lods2pmap_jvierstra
  --cpus-per-task="${NUM_CPU}"
  --mem="${NUM_MEM}"
  --output="${FP_LOG}"
  --chdir="${FD_EXE}"
  --export=ALL,FP_CNF="${FP_CNF}"
  --parsable
)

### execute
SLURM_JOBID=$(sbatch "${SLURM_OPTS[@]}" <<EOF
#!/bin/bash
set -euo pipefail

### init
timer_start=\$(date +%s)
source "${FP_CNF}"

### print start message
echo "Hostname:   \$(hostname)"
echo "Time Stamp: \$(date +"%m-%d-%y+%T")" 
echo "PWD:    \$(pwd)"
echo "FP_APP: ${FP_APP}"
echo "FD_EXE: ${FD_EXE}"
echo "PYTHONPATH (host): \${PYTHONPATH:-<empty>}"
echo

### execute
echo "=== Run main script ==="
${FP_APP} python ${FP_EXE} \
    --txt_fpath_inp       "${TXT_FPATH_INP}" \
    --txt_fpath_out_pmap  "${TXT_FPATH_OUT_PMAP}" \
    --txt_fpath_out_tbind "${TXT_FPATH_OUT_BIND}" \
    --num_workers 50

### print end message
timer=\$(date +%s)
runtime=\$(( timer - timer_start ))
echo
echo 'Done!'
echo "Run Time: \$(displaytime \${runtime})"
EOF
)

echo "Submitted job: ${SLURM_JOBID}"

Submitted job: 43673916


### Review

In [13]:
sacct_summary.sh ${SLURM_JOBID}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43673916.ba+                          batch  COMPLETED   00:00:09  00:09.284    499864K                 dcc-biostat-18 

===== ElapsedRaw =====
ElapsedRaw = 9 sec (0.15 min)

===== MaxRSS =====
MaxRSS = 0.48 GiB


In [14]:
cat ${FD_LOG}/run_motif_lods2pmap_jvierstra.txt

Hostname:   dcc-biostat-18
Time Stamp: 02-23-26+18:19:12
PWD:    /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
FP_APP: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_script.sh
FD_EXE: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PYTHONPATH (host): <empty>

=== Run main script ===
Loaded /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Loaded 637 motifs
Loaded background: [0.31068275 0.18781467 0.18880767 0.31269491]
[100/637] AC0102:ZNF:C2H2_ZF AC0102:ZNF:C2H2_ZF
[200/637] AC0202:ZNF:C2H2_ZF AC0202:ZNF:C2H2_ZF
[300/637] AC0304:ZFP:C2H2_ZF AC0304:ZFP:C2H2_ZF
[400/637] AC0398:POU6F:Homeodomain AC0398:POU6F:Homeodomain
[500/637] AC0502:ZNF:C2H2_ZF AC0502:ZNF:C2H2_ZF
[600/637] AC0598:PAX:Homeodomain,Paired_box AC0598:PAX:Homeodomain,Paired_box
[637/637] AC0637:AHR:bHLH AC0637:AHR:bHLH
Finished precomputing p-maps for 637/637 motifs.
Sanity motif: AC0001:GATA/PROP:GATA AC0001:G

In [15]:
${FP_APP} python - <<'PY'
import pickle, numpy as np

fp_pmap = "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.pmap.pkl"
fp_tbind = "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.tbind.pkl"

with open(fp_pmap,"rb") as f: pmap = pickle.load(f)
with open(fp_tbind,"rb") as f: tb = pickle.load(f)

print("pmap keys:", pmap.keys())
print("meta:", pmap["meta"])
print("num motifs:", len(pmap["motifs"]))
k = next(iter(pmap["motifs"]))
m = pmap["motifs"][k]
print("example:", k, "grid", len(m["arr_num_score_grid"]), "pmf sum", np.sum(m["arr_num_score_pmf"]), "ccdf[0]", m["arr_num_score_ccdf"][0], "tbind", m["num_Tbind"])

print("tbind keys:", tb.keys())
print("num tbind:", len(tb["tbind"]))
PY

pmap keys: dict_keys(['meta', 'motifs'])
meta: {'alphabet': 'ACGT', 'bg': array([0.31068275, 0.18781467, 0.18880767, 0.31269491]), 'alpha': 0.001, 'precision': 0.01, 'score_range': (-60, 60)}
num motifs: 637
example: AC0001:GATA/PROP:GATA AC0001:GATA/PROP:GATA grid 12001 pmf sum 1.0 ccdf[0] 1.0 tbind 6.889999999986685
tbind keys: dict_keys(['meta', 'tbind'])
num tbind: 637
